# HSS-SHAKE256 signed boot

This uses signed/encrypted variants of the **repository-generated IPI design**. No eFUSE writer or persistent enrollment is run. Timing here is host image verification and whole hardware-session wall time; **no BootROM authentication/decryption latency counter has been instrumented**. Do not compare session wall time to runtime crypto latency.

Set BUILD_BOOT=True to package fresh images from the configured runtime build with fresh HSS signing state. Never copy, rewind or reuse private signing state. Set LIVE=True to program the selected image on the exact configured SCU35 and verify programming/recovery status. Fresh hardware acceptance alone is not proof of owner-pinned trust policy; unsigned images were also accepted in earlier tests.

For encrypted combo/rolling images, the diagnostic root is public all-zero, selected from the existing eFUSE key source without writing it. Acceptance depends on existing device provisioning. This proves no secrecy and does not read out or establish default AES key values.

**Saved execution:** 2026-10-26T03:35:48.427452+00:00 (UTC). **fresh hardware + fresh Python**. Hardware cells were executed with `LIVE=True`. Portable defaults below are restored to `LIVE=False`; saved outputs are retained. To repeat on your board, configure it and enable the live flag. Boot outputs show acceptance/recovery and host verification times, not on-chip boot latency.

In [1]:
from pathlib import Path
import sys, os, json, time, hashlib
import pandas as pd
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in (base, base/'scu35') if (p/'lab.py').is_file()), None)
assert ROOT is not None, 'Open the notebook from the repository checkout'
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT/'host'))
from lab import config, tool
from notebook import runtime, show_runtime, LiveRows, command, hardware, check_build
c = config()
OUT = Path(c['build_dir']).resolve()
print('Python:', sys.executable)
print('Build:', OUT)


Python: C:\aiup\pubvenv\Scripts\python.exe
Build: C:\aiup\s35p2


In [2]:
BUILD_BOOT = False
if BUILD_BOOT:
    check_build(c)
    command('boot', c=c)  # H15 default; may take time; creates a NEW signing-state directory
BOOT_DIR = None  # Set to the fresh <build_dir>/boot_TIMESTAMP directory
if BUILD_BOOT:
    BOOT_DIR = max(OUT.glob('boot_*'), key=lambda p:p.stat().st_mtime)
BUNDLE = Path(BOOT_DIR) if BOOT_DIR else ROOT/'evidence/boot/bundle'
print('Fresh build bundle:' if BOOT_DIR else 'Recorded public bundle:', BUNDLE)


Fresh build bundle: C:\aiup\s35p2\boot_20261025T142507


In [3]:
sys.path.insert(0, str(ROOT/'scripts'))
from pdi import verify_signed
PDI = (Path(BOOT_DIR)/'signed.pdi') if BOOT_DIR else ROOT/'reference/boot/signed.pdi'
verification_times = []
for _ in range(3):
    started = time.perf_counter_ns()
    verification, pieces = verify_signed(PDI.read_bytes(), BUNDLE)
    verification_times.append((time.perf_counter_ns()-started)/1e6)
assert all(verification[k] for k in ['spk_signature_valid','hash_block_signature_valid','boot_header_hash_valid'])
display(pd.DataFrame([{'method':'Python full signed-PDI verification (including file read)',
                        'run':i+1,'host_ms':v} for i,v in enumerate(verification_times)]))
display(pd.DataFrame([verification]))


,method,run,host_ms
0,Python full signed-PDI verification (including...,1,28.0463
1,Python full signed-PDI verification (including...,2,10.9465
2,Python full signed-PDI verification (including...,3,10.3457


,header,ppk_sha256,spk_sha256,spk_signature_valid,hash_block_signature_valid,boot_header_hash_valid,signature_bytes,signature_offset,hash_block_offset,plm_sha256,pl_sha256,plm_bytes,pl_bytes,chain_chunks
0,"{'key_source': 0, 'plm_offset': 19120, 'pl_loa...",6861e5df0680eae6630b529463e0829e99e1a9f48d2467...,b6ea08c1c27f68e75df768b7eb8a19188cc3e0c1f7c083...,True,True,True,8980,10128,9984,bdafedd897fe86e0f078cf76339baaa0f18826292efc04...,d7a33ea02987eb16953bc1a9ea1e060c4d90114c57d22f...,26048,1327536,167


In [4]:
LIVE = False  # Explicitly program the selected image and then recover
if LIVE:
    assert BOOT_DIR is not None, 'Choose variants of your freshly generated repository IPI build'
    session, session_wall_s = hardware(PDI, mode='boot', c=c)
    display(pd.DataFrame([{'session':str(session),'program_capture_recovery_wall_s':session_wall_s,
                           'on_chip_boot_crypto_us':None,
                           'scope':'Whole host session; not instrumented boot crypto latency'}]))
else:
    print('No hardware accessed. Python verification above was freshly timed.')


RUN: hardware --mode boot --pdi C:\aiup\s35p2\boot_20261025T142507\signed.pdi


RUN C:\Xilinx\2025.2\Vivado\bin\vivado.bat


HARDWARE_SESSION_PASS C:\aiup\s35p2\runs\20261025T213550


Fresh hardware and recovery PASS: C:\aiup\s35p2\runs\20261025T213550


Host session wall time (program/capture/recovery, NOT crypto latency): 43.67821100004949 s


,session,program_capture_recovery_wall_s,on_chip_boot_crypto_us,scope
0,C:\aiup\s35p2\runs\20261025T213550,43.678211,None,Whole host session; not instrumented boot cryp...
